# Combining the Z cross section from $\mu\mu$ and $\tau_h\tau_h$

The two finished BND-school channels measure the same quantity on the same CMS Open Data 2016
sample (Run2016G+H, $L = 16393.381$ pb$^{-1}$). This notebook combines them.

It is a **thin wrapper**: all the physics is in `comb/`, the same code `run_combination.py`
runs, so the notebook and `result.md` can never disagree. Read
[`docs/00-overview.md`](docs/00-overview.md) first; `z-ee` is out of scope.

```bash
source ../setup.sh && jupyter notebook combination.ipynb
```


In [ ]:
import sys; sys.path.insert(0, '.')
from comb import inputs, model, blue, likelihood, ratio

import matplotlib.pyplot as plt


## 1. Inputs

Read from the channels' committed TRExFitter results -- nothing is refitted. Two choices are
made here and justified in [`docs/01-inputs.md`](docs/01-inputs.md):

* `mumu="counting"`: the extraction `z-mumu/REVIEW.md` (F3) recommends, since the 30-bin shape
  fit moves $\mu_Z$ by up to 1.5 % with the binning. It carries an extra $\pm0.7$ % lineshape term.
* `tautau="nominal"`: the fake-factor variant that channel published.

Note the `reference` column: the two channels normalise to aMC@NLO predictions of the *same*
quantity that differ by 0.47 %. That is why we combine cross sections and never $\mu_Z$.


In [ ]:
chans = inputs.load_channels(mumu='counting', tautau='nominal')

for c in chans.values():
    print(f'{c.name:>7s} ({c.variant:<8s})  mu_Z = {c.mu:6.4f}   reference = {c.sigma_pred:7.1f} pb'
          f'   sigma = {c.sigma:7.1f} +- {c.sigma_err_total:5.1f} pb')
    print('           provenance:', ', '.join(c.provenance))


## 2. The correlation model

The only judgement in the whole folder. Keyed on the `Category` strings that
`../fitting/CONVENTIONS.md` forces both channels to use; a category with no assigned
correlation raises rather than defaulting to zero. Every row is justified in
[`docs/02-correlation-model.md`](docs/02-correlation-model.md).


In [ ]:
spec = model.build(chans)

print(f"{'source':<38s} {'rho':>4s} {'mumu [pb]':>10s} {'tautau [pb]':>12s}")
for s in sorted(spec.sources, key=lambda s: -max(s.sizes.values())):
    print(f'{s.name:<38s} {s.rho:4.1f} {s.sizes.get("mumu", 0):10.2f} {s.sizes.get("tautau", 0):12.2f}')


In [ ]:
import numpy as np

cov = spec.covariance()
d = np.sqrt(np.diag(cov))
print('covariance [pb^2]:\n', np.array2string(cov, precision=1))
print(f'\nrho(mumu, tautau) = {cov[0, 1] / (d[0] * d[1]):.3f}')


## 3. Combination (BLUE)

$w = C^{-1}u/(u^TC^{-1}u)$, $\hat\sigma = w^Tx$, $\mathrm{Var} = 1/(u^TC^{-1}u)$, and
$\chi^2$ with ndf = 1 for the compatibility of the two channels. The acceptance terms are
multiplicative, so the solution is iterated to remove the normalisation bias.


In [ ]:
res = blue.combine(spec)
solo = {n: blue.single(spec, n) for n in spec.order}
g = res.group_breakdown()

print(f'sigma(Z/gamma* -> ll, 60 < m < 120 GeV) = {res.value:.0f} +- {res.error:.0f} pb'
      f'  ({100 * res.rel:.2f} %)')
print(f'   = {res.value:.0f} +- {g["statistical"]:.1f} (stat) +- {g["other systematic"]:.0f} (syst)'
      f' +- {g["acceptance"]:.0f} (acc) +- {g["luminosity"]:.0f} (lumi) pb')
print(f'weights : ' + ', '.join(f'{k} {v:+.4f}' for k, v in res.weights.items()))
print(f'chi2/ndf = {res.chi2:.2f}/{res.ndf}, p = {res.pvalue:.3f}')
print(f'mumu alone: {solo["mumu"].value:.0f} +- {solo["mumu"].error:.2f} pb'
      f'  ->  combining gains {100 * (1 - res.error / solo["mumu"].error):.2f} %')


The $\tau\tau$ weight is **negative**. That is standard BLUE behaviour once the correlation
exceeds the ratio of the two uncertainties: the less precise measurement stops averaging the
value down and becomes a lever on the shared systematic. It moves the central value by 1.5 pb.

### Where the uncertainty comes from


In [ ]:
for name, value in res.breakdown.items():
    if value >= 0.05:
        print(f'  {name:<38s} {value:7.2f} pb   ({100 * value / res.value:5.2f} %)')


Luminosity is 24 of the 35 pb and is **100 % correlated between the channels**, so it cannot be
combined away. This is why the combination gains almost nothing, and it is the main result of
the exercise: the way to a better number here is a better luminosity calibration, not more channels.


## 4. Profile-likelihood cross-check

The same combination written as an explicit likelihood with one nuisance parameter per shared
source. For symmetric errors it is algebraically identical to BLUE; it additionally carries the
$\tau\tau$ channel's asymmetric uncertainty (+16.6 / -14.1 %) through a bifurcated response.


In [ ]:
lik = likelihood.combine(spec, asymmetric=True)
print(f'profile likelihood : {lik.value:.1f} +{lik.error_up:.1f} -{lik.error_down:.1f} pb')
print(f'BLUE               : {res.value:.1f} +- {res.error:.1f} pb')

grid, curve = lik.scan
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(grid, curve, lw=2, label='profile likelihood')
ax.plot(grid, ((grid - res.value) / res.error) ** 2, ls='--', lw=1.4, label='BLUE parabola')
ax.axhline(1, color='grey', ls=':'); ax.set_ylim(0, 9)
ax.set_xlabel(r'$\sigma(60<m<120)$ [pb]'); ax.set_ylabel(r'$-2\Delta\ln L$')
ax.legend(); plt.show()


## 5. Lepton universality

The combination *assumes* universality, so it cannot test it. The ratio can: luminosity,
pileup, prefiring and the correlated theory terms cancel exactly.


In [ ]:
rat = ratio.compute(spec)
print(f'R = sigma(tautau)/sigma(mumu) = {rat.value:.3f} +- {rat.error:.3f}'
      f'   ({rat.z_from_unity:+.2f} sigma from 1, p = {rat.pvalue:.3f})')
print()
for name, value in list(rat.breakdown.items())[:8]:
    print(f'  {name:<38s} {value:.3f}')


## 6. Everything else

The variations (ττ `mcsub`, μμ shape fit, ρ = 0, ρ = 1, ...), the figures and `result.md` come
from the orchestrator, which runs the checks first:

```bash
python run_combination.py
```

Outputs: [`result.md`](result.md), `output/combination_result.json`, `output/plots/*.pdf`.
